# Northwind Tester 1 — Problems and Solutions

Đây là workbook thực hành Data Cleaning hoàn chỉnh. Bạn sẽ mở raw SQLite database, quan sát từng vấn đề bằng code, chạy lời giải preprocessing, rồi tạo `northwind_cleaned1.db`.

Notebook có 12 bài. Mỗi bài luôn đi theo thứ tự: **Problem → Detect → Solution → Check**. Toàn bộ code nằm trong code cell; Markdown chỉ giải thích ý nghĩa.

Quy tắc an toàn: raw database chỉ được đọc. Mọi thay đổi diễn ra trên một working database trong RAM và chỉ được lưu thành clean output sau khi validation đạt

## 0. Chuẩn bị môi trường

Cell dưới chỉ dùng Python standard library nên không cần pandas hoặc package ngoài. Đường dẫn được xác định tương đối để notebook chạy được cả khi mở từ folder tester1 hoặc từ project root.

In [60]:
from pathlib import Path
import hashlib
import json
import os
import re
import sqlite3
import tempfile
from datetime import datetime

cwd = Path.cwd()
FOLDER = cwd if (cwd / 'northwind_tester1.db').exists() else cwd / 'data_raw_tester1'
RAW = FOLDER / 'northwind_tester1.db'
GROUND_TRUTH = FOLDER / 'revert_clean_tester1.json'
CLEAN = FOLDER / 'northwind_cleaned1.db'

with GROUND_TRUTH.open(encoding='utf-8') as stream:
    bundle = json.load(stream)

def sha256_file(path):
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

assert RAW.exists(), f'Không tìm thấy raw database: {RAW}'
assert sha256_file(RAW) == bundle['dataset']['raw_sha256'], 'Raw checksum không đúng'
print('Raw input :', RAW.name)
print('Clean output:', CLEAN.name)
print('Ground-truth faults:', len(bundle['repair_records']))
print('Clean controls:', len(bundle['clean_controls']))

Raw input : northwind_tester1.db
Clean output: northwind_cleaned1.db
Ground-truth faults: 5170
Clean controls: 5170


## 1. Khám phá raw database

Trước khi cleaning, luôn kiểm tra database mở được, integrity còn tốt, các foreign key không bị vỡ và quy mô từng bảng. Data quality không chỉ là NULL; một database có thể hợp lệ về cấu trúc nhưng vẫn chứa giá trị sai nghĩa.

In [61]:
raw_connection = sqlite3.connect(RAW.resolve().as_uri() + '?mode=ro', uri=True)
raw_connection.row_factory = sqlite3.Row

tables = [row['name'] for row in raw_connection.execute(
    "SELECT name FROM sqlite_master WHERE type='table' "
    "AND name NOT LIKE 'sqlite_%' ORDER BY name"
)]
print('Integrity:', raw_connection.execute('PRAGMA integrity_check').fetchone()[0])
print('Foreign-key violations:', len(raw_connection.execute('PRAGMA foreign_key_check').fetchall()))
print('\nTable row counts:')
for table in tables:
    safe_table = '"' + table.replace('"', '""') + '"'
    count = raw_connection.execute(f'SELECT COUNT(*) FROM {safe_table}').fetchone()[0]
    print(f'  {table:24} {count:>8,}')
raw_connection.close()

Integrity: ok
Foreign-key violations: 0

Table row counts:
  Categories                      8
  CustomerCustomerDemo            0
  CustomerDemographics            0
  Customers                     113
  EmployeeTerritories            49
  Employees                       9
  Order Details             609,283
  Orders                     16,282
  Products                       77
  Regions                         4
  Shippers                        3
  Suppliers                      29
  Territories                    53


## 2. Tạo working database trong RAM

Cell này copy raw database vào memory. Từ đây trở đi, các câu lệnh UPDATE và DELETE chỉ tác động working copy. Các helper được viết ngay trong notebook để người học nhìn thấy toàn bộ cơ chế; notebook không import lời giải từ file Python.

In [62]:
def connect_read_only(path):
    connection = sqlite3.connect(path.resolve().as_uri() + '?mode=ro', uri=True)
    connection.row_factory = sqlite3.Row
    return connection

source = connect_read_only(RAW)
connection = sqlite3.connect(':memory:')
connection.row_factory = sqlite3.Row
source.backup(connection)
source.close()
connection.execute('PRAGMA foreign_keys = ON')

DETECTION_SQL = {
    item['rule_id']: item['detection']['candidate_sql']
    for item in bundle['preprocessing_plan']
}

def quote_name(name):
    return '"' + name.replace('"', '""') + '"'

def records_for(rule_id):
    return [record for record in bundle['repair_records'] if record['rule_id'] == rule_id]

def primary_key_filter(primary_key):
    clause = ' AND '.join(f'{quote_name(column)} IS ?' for column in primary_key)
    return clause, list(primary_key.values())

def find_row(table, primary_key):
    where, parameters = primary_key_filter(primary_key)
    return connection.execute(
        f'SELECT * FROM {quote_name(table)} WHERE {where}', parameters
    ).fetchone()

def unresolved_count(rule_id):
    unresolved = 0
    for record in records_for(rule_id):
        row = find_row(record['table'], record['primary_key'])
        if record['operation'] == 'insert_row':
            unresolved += int(row is not None)
        else:
            unresolved += int(row is None or row[record['column']] != record['clean_value'])
    return unresolved

def show_candidates(rule_id, limit=5):
    query = DETECTION_SQL[rule_id].strip().rstrip(';')
    count = connection.execute(f'SELECT COUNT(*) FROM ({query})').fetchone()[0]
    rows = [dict(row) for row in connection.execute(query).fetchmany(limit)]
    print(f'Candidate rows: {count:,}')
    print(f'Exact ground-truth faults: {len(records_for(rule_id)):,}')
    print('Raw examples:')
    for row in rows:
        print(' ', row)
    return count

def show_repaired_examples(rule_id, limit=5):
    print('After cleaning examples:')
    for record in records_for(rule_id)[:limit]:
        row = find_row(record['table'], record['primary_key'])
        if record['operation'] == 'insert_row':
            cleaned_value = '<row deleted>' if row is None else '<row still exists>'
            matches_ground_truth = row is None
        else:
            cleaned_value = None if row is None else row[record['column']]
            matches_ground_truth = cleaned_value == record['clean_value']
        print(' ', {
            'primary_key': record['primary_key'],
            'raw_value': record['corrupted_value'],
            'cleaned_value': cleaned_value,
            'expected_clean': record['clean_value'],
            'matches_ground_truth': matches_ground_truth,
        })

def apply_transform(rule_id, transform):
    before, updated = unresolved_count(rule_id), 0
    for record in records_for(rule_id):
        row = find_row(record['table'], record['primary_key'])
        if row is None:
            raise RuntimeError(f'Missing row: {record["primary_key"]}')
        raw_value = row[record['column']]
        clean_value = transform(raw_value, record)
        assert clean_value == record['clean_value'], (
            rule_id, raw_value, clean_value, record['clean_value'])
        where, parameters = primary_key_filter(record['primary_key'])
        connection.execute(
            f'UPDATE {quote_name(record["table"])} '
            f'SET {quote_name(record["column"])} = ? WHERE {where}',
            [clean_value, *parameters],
        )
        updated += 1
    report = {'rule': rule_id, 'before': before, 'updated': updated,
              'after': unresolved_count(rule_id)}
    print(report)
    return report

def restore_supervised_labels(rule_id, reason):
    before, updated = unresolved_count(rule_id), 0
    for record in records_for(rule_id):
        where, parameters = primary_key_filter(record['primary_key'])
        cursor = connection.execute(
            f'UPDATE {quote_name(record["table"])} '
            f'SET {quote_name(record["column"])} = ? WHERE {where}',
            [record['clean_value'], *parameters],
        )
        assert cursor.rowcount == 1
        updated += 1
    report = {'rule': rule_id, 'method': 'supervised_ground_truth',
              'reason': reason, 'before': before, 'updated': updated,
              'after': unresolved_count(rule_id)}
    print(report)
    return report

print('Working copy is ready in memory.')

Working copy is ready in memory.


## Problem 1: Freight được lưu dưới dạng currency text

**Problem.** Orders.Freight đáng lẽ là số, nhưng 700 giá trị có dạng như USD 30 hoặc USD 28.75. Điều này làm phép tính tổng, trung bình và sắp xếp số học không còn đáng tin cậy.

**Detection.** Chạy code cell tiếp theo để xem tổng số candidate và tối đa 5 ví dụ raw. Candidate detection phục vụ khám phá; exact ground-truth faults mới quyết định các dòng được reference answer sửa.

In [63]:
# Detect Problem 1
problem_1_candidates = show_candidates('T1-TEXT-FREIGHT', limit=5)

Candidate rows: 700
Exact ground-truth faults: 700
Raw examples:
  {'OrderID': 10269, 'Freight': 'USD 30', 'storage_type': 'text'}
  {'OrderID': 10294, 'Freight': 'USD 28.75', 'storage_type': 'text'}
  {'OrderID': 10310, 'Freight': 'USD 13.75', 'storage_type': 'text'}
  {'OrderID': 10324, 'Freight': 'USD 70.25', 'storage_type': 'text'}
  {'OrderID': 10344, 'Freight': 'USD 36.25', 'storage_type': 'text'}


### Solution 1

Loại bỏ ký hiệu không thuộc số, parse phần còn lại thành float, rồi trả về int khi giá trị không có phần thập phân. Đây là deterministic repair vì raw vẫn giữ đủ giá trị.

In [64]:
def parse_freight(value, record):
    number = float(re.sub(r'[^0-9,.-]', '', str(value)).replace(',', '.'))
    return int(number) if number.is_integer() else number

problem_1_report = apply_transform('T1-TEXT-FREIGHT', parse_freight)
show_repaired_examples('T1-TEXT-FREIGHT')

{'rule': 'T1-TEXT-FREIGHT', 'before': 700, 'updated': 700, 'after': 0}
After cleaning examples:
  {'primary_key': {'OrderID': 10269}, 'raw_value': 'USD 30', 'cleaned_value': 30, 'expected_clean': 30, 'matches_ground_truth': True}
  {'primary_key': {'OrderID': 10294}, 'raw_value': 'USD 28.75', 'cleaned_value': 28.75, 'expected_clean': 28.75, 'matches_ground_truth': True}
  {'primary_key': {'OrderID': 10310}, 'raw_value': 'USD 13.75', 'cleaned_value': 13.75, 'expected_clean': 13.75, 'matches_ground_truth': True}
  {'primary_key': {'OrderID': 10324}, 'raw_value': 'USD 70.25', 'cleaned_value': 70.25, 'expected_clean': 70.25, 'matches_ground_truth': True}
  {'primary_key': {'OrderID': 10344}, 'raw_value': 'USD 36.25', 'cleaned_value': 36.25, 'expected_clean': 36.25, 'matches_ground_truth': True}


## Problem 2: Product price được lưu dưới dạng currency text

**Problem.** Products.UnitPrice là cột numeric nhưng một nhóm sản phẩm đang chứa text có tiền tố tiền tệ. Đây là cùng loại lỗi storage type với Freight nhưng nằm ở bảng khác.

**Detection.** Chạy code cell tiếp theo để xem tổng số candidate và tối đa 5 ví dụ raw. Candidate detection phục vụ khám phá; exact ground-truth faults mới quyết định các dòng được reference answer sửa.

In [65]:
# Detect Problem 2
problem_2_candidates = show_candidates('T1-TEXT-PRODUCT-PRICE', limit=5)

Candidate rows: 30
Exact ground-truth faults: 30
Raw examples:
  {'ProductID': 3, 'ProductName': '  Aniseed  Syrup   ', 'UnitPrice': '$10', 'storage_type': 'text'}
  {'ProductID': 5, 'ProductName': "  Chef  Anton's  Gumbo  Mix   ", 'UnitPrice': '$21.35', 'storage_type': 'text'}
  {'ProductID': 7, 'ProductName': "Uncle Bob's Organic Dried Pears", 'UnitPrice': '$30', 'storage_type': 'text'}
  {'ProductID': 9, 'ProductName': '  Mishi  Kobe  Niku   ', 'UnitPrice': '$97', 'storage_type': 'text'}
  {'ProductID': 13, 'ProductName': 'Konbu', 'UnitPrice': '$6', 'storage_type': 'text'}


### Solution 2

Tách phần numeric khỏi chuỗi và lưu lại bằng SQLite numeric storage. Giá trị được suy ra trực tiếp từ raw.

In [66]:
def parse_product_price(value, record):
    number = float(re.sub(r'[^0-9,.-]', '', str(value)).replace(',', '.'))
    return int(number) if number.is_integer() else number

problem_2_report = apply_transform('T1-TEXT-PRODUCT-PRICE', parse_product_price)
show_repaired_examples('T1-TEXT-PRODUCT-PRICE')

{'rule': 'T1-TEXT-PRODUCT-PRICE', 'before': 30, 'updated': 30, 'after': 0}
After cleaning examples:
  {'primary_key': {'ProductID': 13}, 'raw_value': '$6', 'cleaned_value': 6, 'expected_clean': 6, 'matches_ground_truth': True}
  {'primary_key': {'ProductID': 14}, 'raw_value': '$23.25', 'cleaned_value': 23.25, 'expected_clean': 23.25, 'matches_ground_truth': True}
  {'primary_key': {'ProductID': 15}, 'raw_value': '$15.5', 'cleaned_value': 15.5, 'expected_clean': 15.5, 'matches_ground_truth': True}
  {'primary_key': {'ProductID': 16}, 'raw_value': '$17.45', 'cleaned_value': 17.45, 'expected_clean': 17.45, 'matches_ground_truth': True}
  {'primary_key': {'ProductID': 22}, 'raw_value': '$21', 'cleaned_value': 21, 'expected_clean': 21, 'matches_ground_truth': True}


## Problem 3: Discount trộn percentage text với fraction

**Problem.** Order Details.Discount bình thường dùng miền từ 0 đến 1, nhưng 500 ô được ghi như 15%. Nếu không chuẩn hóa, cùng một cột sẽ chứa hai cách biểu diễn khác nhau.

**Detection.** Chạy code cell tiếp theo để xem tổng số candidate và tối đa 5 ví dụ raw. Candidate detection phục vụ khám phá; exact ground-truth faults mới quyết định các dòng được reference answer sửa.

In [67]:
# Detect Problem 3
problem_3_candidates = show_candidates('T1-PERCENT-DISCOUNT', limit=5)

Candidate rows: 500
Exact ground-truth faults: 500
Raw examples:
  {'OrderID': 10250, 'ProductID': 65, 'Discount': '15%', 'storage_type': 'text'}
  {'OrderID': 10251, 'ProductID': 22, 'Discount': '5%', 'storage_type': 'text'}
  {'OrderID': 10252, 'ProductID': 20, 'Discount': '5%', 'storage_type': 'text'}
  {'OrderID': 10252, 'ProductID': 33, 'Discount': '5%', 'storage_type': 'text'}
  {'OrderID': 10254, 'ProductID': 24, 'Discount': '15%', 'storage_type': 'text'}


### Solution 3

Bỏ ký tự phần trăm, chuyển sang float và chia cho 100. Ví dụ 15% trở thành 0.15.

In [68]:
def parse_discount(value, record):
    return float(str(value).strip().removesuffix('%')) / 100

problem_3_report = apply_transform('T1-PERCENT-DISCOUNT', parse_discount)
show_repaired_examples('T1-PERCENT-DISCOUNT')

{'rule': 'T1-PERCENT-DISCOUNT', 'before': 500, 'updated': 500, 'after': 0}
After cleaning examples:
  {'primary_key': {'OrderID': 10250, 'ProductID': 65}, 'raw_value': '15%', 'cleaned_value': 0.15, 'expected_clean': 0.15, 'matches_ground_truth': True}
  {'primary_key': {'OrderID': 10251, 'ProductID': 22}, 'raw_value': '5%', 'cleaned_value': 0.05, 'expected_clean': 0.05, 'matches_ground_truth': True}
  {'primary_key': {'OrderID': 10252, 'ProductID': 20}, 'raw_value': '5%', 'cleaned_value': 0.05, 'expected_clean': 0.05, 'matches_ground_truth': True}
  {'primary_key': {'OrderID': 10252, 'ProductID': 33}, 'raw_value': '5%', 'cleaned_value': 0.05, 'expected_clean': 0.05, 'matches_ground_truth': True}
  {'primary_key': {'OrderID': 10254, 'ProductID': 24}, 'raw_value': '15%', 'cleaned_value': 0.15, 'expected_clean': 0.15, 'matches_ground_truth': True}


## Problem 4: OrderDate có nhiều định dạng

**Problem.** Orders.OrderDate chứa ba format không theo chuẩn ISO. Hai format chỉ giữ tới phút nên phần giây ban đầu đã bị mất.

**Detection.** Chạy code cell tiếp theo để xem tổng số candidate và tối đa 5 ví dụ raw. Candidate detection phục vụ khám phá; exact ground-truth faults mới quyết định các dòng được reference answer sửa.

In [69]:
# Detect Problem 4
problem_4_candidates = show_candidates('T1-MIXED-ORDER-DATE', limit=5)

Candidate rows: 700
Exact ground-truth faults: 700
Raw examples:
  {'OrderID': 10275, 'OrderDate': '07/08/2016 00:00'}
  {'OrderID': 10284, 'OrderDate': '08-19-2016 00:00:00'}
  {'OrderID': 10290, 'OrderDate': '2016/08/27 00:00'}
  {'OrderID': 10310, 'OrderDate': '20/09/2016 00:00'}
  {'OrderID': 10342, 'OrderDate': '10-30-2016 00:00:00'}


### Solution 4

Thử parse lần lượt ba format. Format giữ đủ giây được sửa hoàn toàn từ raw. Với format mất giây, code xác minh phần ngày và phút rồi lấy riêng precision đã mất từ supervised label.

In [70]:
def parse_order_date(value, record):
    raw, label = str(value), str(record['clean_value'])
    formats = ('%d/%m/%Y %H:%M', '%m-%d-%Y %H:%M:%S', '%Y/%m/%d %H:%M')
    for date_format in formats:
        try:
            parsed = datetime.strptime(raw, date_format)
        except ValueError:
            continue
        if '%S' in date_format:
            full = parsed.strftime('%Y-%m-%d %H:%M:%S')
            return full if ' ' in label else full[:10]
        retained = parsed.strftime('%Y-%m-%d %H:%M')
        assert label == retained[:10] or label.startswith(retained)
        return label
    raise ValueError(f'Unsupported date: {raw}')

problem_4_report = apply_transform('T1-MIXED-ORDER-DATE', parse_order_date)
show_repaired_examples('T1-MIXED-ORDER-DATE')

{'rule': 'T1-MIXED-ORDER-DATE', 'before': 700, 'updated': 700, 'after': 0}
After cleaning examples:
  {'primary_key': {'OrderID': 10275}, 'raw_value': '07/08/2016 00:00', 'cleaned_value': '2016-08-07', 'expected_clean': '2016-08-07', 'matches_ground_truth': True}
  {'primary_key': {'OrderID': 10284}, 'raw_value': '08-19-2016 00:00:00', 'cleaned_value': '2016-08-19', 'expected_clean': '2016-08-19', 'matches_ground_truth': True}
  {'primary_key': {'OrderID': 10290}, 'raw_value': '2016/08/27 00:00', 'cleaned_value': '2016-08-27', 'expected_clean': '2016-08-27', 'matches_ground_truth': True}
  {'primary_key': {'OrderID': 10310}, 'raw_value': '20/09/2016 00:00', 'cleaned_value': '2016-09-20', 'expected_clean': '2016-09-20', 'matches_ground_truth': True}
  {'primary_key': {'OrderID': 10342}, 'raw_value': '10-30-2016 00:00:00', 'cleaned_value': '2016-10-30', 'expected_clean': '2016-10-30', 'matches_ground_truth': True}


## Problem 5: Phone bị thay bằng missing-value placeholder

**Problem.** Customers.Phone có các token N/A, unknown, dấu gạch ngang hoặc chuỗi rỗng. Các token cho biết dữ liệu bị thiếu nhưng không chứa số điện thoại gốc.

**Detection.** Chạy code cell tiếp theo để xem tổng số candidate và tối đa 5 ví dụ raw. Candidate detection phục vụ khám phá; exact ground-truth faults mới quyết định các dòng được reference answer sửa.

In [71]:
# Detect Problem 5
problem_5_candidates = show_candidates('T1-PHONE-PLACEHOLDERS', limit=5)

Candidate rows: 46
Exact ground-truth faults: 40
Raw examples:
  {'CustomerID': 'ALFKI', 'Phone': 'N/A'}
  {'CustomerID': 'ANTON', 'Phone': 'unknown'}
  {'CustomerID': 'AROUT', 'Phone': '-'}
  {'CustomerID': 'BLONP', 'Phone': 'N/A'}
  {'CustomerID': 'BSBEV', 'Phone': 'unknown'}


### Solution 5

Detection thực hiện từ raw. Muốn tái tạo clean database chính xác, answer sheet phải dùng clean_value trong supervised JSON. Đây là oracle repair được công khai, không phải giá trị tự suy luận.

In [72]:
problem_5_report = restore_supervised_labels(
    'T1-PHONE-PLACEHOLDERS',
    'Placeholder cho biết missingness nhưng không cho biết số điện thoại gốc.'
)
show_repaired_examples('T1-PHONE-PLACEHOLDERS')

{'rule': 'T1-PHONE-PLACEHOLDERS', 'method': 'supervised_ground_truth', 'reason': 'Placeholder cho biết missingness nhưng không cho biết số điện thoại gốc.', 'before': 40, 'updated': 40, 'after': 0}
After cleaning examples:
  {'primary_key': {'CustomerID': 'ALFKI'}, 'raw_value': 'N/A', 'cleaned_value': '030-0074321', 'expected_clean': '030-0074321', 'matches_ground_truth': True}
  {'primary_key': {'CustomerID': 'ANTON'}, 'raw_value': 'unknown', 'cleaned_value': '(5) 555-3932', 'expected_clean': '(5) 555-3932', 'matches_ground_truth': True}
  {'primary_key': {'CustomerID': 'AROUT'}, 'raw_value': '-', 'cleaned_value': '(171) 555-7788', 'expected_clean': '(171) 555-7788', 'matches_ground_truth': True}
  {'primary_key': {'CustomerID': 'BLONP'}, 'raw_value': 'N/A', 'cleaned_value': '88.60.15.31', 'expected_clean': '88.60.15.31', 'matches_ground_truth': True}
  {'primary_key': {'CustomerID': 'BSBEV'}, 'raw_value': 'unknown', 'cleaned_value': '(171) 555-1212', 'expected_clean': '(171) 555-1212

## Problem 6: ShipName có whitespace thừa

**Problem.** Orders.ShipName có leading space, trailing space, tab hoặc nhiều khoảng trắng liên tiếp. Những biến thể này gây sai group, join và duplicate detection.

**Detection.** Chạy code cell tiếp theo để xem tổng số candidate và tối đa 5 ví dụ raw. Candidate detection phục vụ khám phá; exact ground-truth faults mới quyết định các dòng được reference answer sửa.

In [73]:
# Detect Problem 6
problem_6_candidates = show_candidates('T1-WHITESPACE-SHIP-NAME', limit=5)

Candidate rows: 700
Exact ground-truth faults: 700
Raw examples:
  {'OrderID': 10274, 'ShipName': '  Vins  et  alcools  Chevalier   '}
  {'OrderID': 10293, 'ShipName': '  Tortuga  Restaurante   '}
  {'OrderID': 10303, 'ShipName': '  Godos  Cocina  Típica   '}
  {'OrderID': 10332, 'ShipName': '  Mère  Paillarde   '}
  {'OrderID': 10369, 'ShipName': '  Split  Rail  Beer  &  Ale   '}


### Solution 6

split không tham số nhận mọi chuỗi whitespace, sau đó join bằng đúng một dấu cách. Cách này đồng thời trim hai đầu và collapse khoảng trắng bên trong.

In [74]:
def normalize_ship_name(value, record):
    return ' '.join(str(value).split())

problem_6_report = apply_transform('T1-WHITESPACE-SHIP-NAME', normalize_ship_name)
show_repaired_examples('T1-WHITESPACE-SHIP-NAME')

{'rule': 'T1-WHITESPACE-SHIP-NAME', 'before': 700, 'updated': 700, 'after': 0}
After cleaning examples:
  {'primary_key': {'OrderID': 10274}, 'raw_value': '  Vins  et  alcools  Chevalier   ', 'cleaned_value': 'Vins et alcools Chevalier', 'expected_clean': 'Vins et alcools Chevalier', 'matches_ground_truth': True}
  {'primary_key': {'OrderID': 10293}, 'raw_value': '  Tortuga  Restaurante   ', 'cleaned_value': 'Tortuga Restaurante', 'expected_clean': 'Tortuga Restaurante', 'matches_ground_truth': True}
  {'primary_key': {'OrderID': 10303}, 'raw_value': '  Godos  Cocina  Típica   ', 'cleaned_value': 'Godos Cocina Típica', 'expected_clean': 'Godos Cocina Típica', 'matches_ground_truth': True}
  {'primary_key': {'OrderID': 10332}, 'raw_value': '  Mère  Paillarde   ', 'cleaned_value': 'Mère Paillarde', 'expected_clean': 'Mère Paillarde', 'matches_ground_truth': True}
  {'primary_key': {'OrderID': 10369}, 'raw_value': '  Split  Rail  Beer  &  Ale   ', 'cleaned_value': 'Split Rail Beer & Ale',

## Problem 7: ProductName có whitespace thừa

**Problem.** Products.ProductName gặp cùng loại whitespace noise nhưng trên master data sản phẩm. Cần xử lý mà không thay đổi nội dung chữ.

**Detection.** Chạy code cell tiếp theo để xem tổng số candidate và tối đa 5 ví dụ raw. Candidate detection phục vụ khám phá; exact ground-truth faults mới quyết định các dòng được reference answer sửa.

In [75]:
# Detect Problem 7
problem_7_candidates = show_candidates('T1-WHITESPACE-PRODUCT', limit=5)

Candidate rows: 30
Exact ground-truth faults: 30
Raw examples:
  {'ProductID': 3, 'ProductName': '  Aniseed  Syrup   '}
  {'ProductID': 4, 'ProductName': "  Chef  Anton's  Cajun  Seasoning   "}
  {'ProductID': 5, 'ProductName': "  Chef  Anton's  Gumbo  Mix   "}
  {'ProductID': 9, 'ProductName': '  Mishi  Kobe  Niku   '}
  {'ProductID': 10, 'ProductName': '  Ikura   '}


### Solution 7

Trim và collapse toàn bộ whitespace runs về một dấu cách, sau đó đối chiếu exact label.

In [76]:
def normalize_product_name(value, record):
    return ' '.join(str(value).split())

problem_7_report = apply_transform('T1-WHITESPACE-PRODUCT', normalize_product_name)
show_repaired_examples('T1-WHITESPACE-PRODUCT')

{'rule': 'T1-WHITESPACE-PRODUCT', 'before': 30, 'updated': 30, 'after': 0}
After cleaning examples:
  {'primary_key': {'ProductID': 10}, 'raw_value': '  Ikura   ', 'cleaned_value': 'Ikura', 'expected_clean': 'Ikura', 'matches_ground_truth': True}
  {'primary_key': {'ProductID': 14}, 'raw_value': '  Tofu   ', 'cleaned_value': 'Tofu', 'expected_clean': 'Tofu', 'matches_ground_truth': True}
  {'primary_key': {'ProductID': 17}, 'raw_value': '  Alice  Mutton   ', 'cleaned_value': 'Alice Mutton', 'expected_clean': 'Alice Mutton', 'matches_ground_truth': True}
  {'primary_key': {'ProductID': 22}, 'raw_value': "  Gustaf's  Knäckebröd   ", 'cleaned_value': "Gustaf's Knäckebröd", 'expected_clean': "Gustaf's Knäckebröd", 'matches_ground_truth': True}
  {'primary_key': {'ProductID': 23}, 'raw_value': '  Tunnbröd   ', 'cleaned_value': 'Tunnbröd', 'expected_clean': 'Tunnbröd', 'matches_ground_truth': True}


## Problem 8: ShipCity sai quy ước hoa/thường

**Problem.** Orders.ShipCity có tên thành phố bị chuyển toàn bộ sang upper hoặc lower case. Không nên dùng title-case mù quáng vì có tên địa lý và dấu câu đặc biệt.

**Detection.** Chạy code cell tiếp theo để xem tổng số candidate và tối đa 5 ví dụ raw. Candidate detection phục vụ khám phá; exact ground-truth faults mới quyết định các dòng được reference answer sửa.

In [77]:
# Detect Problem 8
problem_8_candidates = show_candidates('T1-CASE-SHIP-CITY', limit=5)

Candidate rows: 847
Exact ground-truth faults: 700
Raw examples:
  {'OrderID': 10249, 'ShipCity': 'münster'}
  {'OrderID': 10294, 'ShipCity': 'ALBUQUERQUE'}
  {'OrderID': 10301, 'ShipCity': 'stuttgart'}
  {'OrderID': 10305, 'ShipCity': 'ANCHORAGE'}
  {'OrderID': 10360, 'ShipCity': 'strasbourg'}


### Solution 8

So sánh case-insensitive để xác nhận chỉ khác capitalization, rồi dùng canonical label trong supervised vocabulary để giữ chính xác dấu và cách viết.

In [78]:
def canonical_city_case(value, record):
    canonical = str(record['clean_value'])
    assert str(value).casefold() == canonical.casefold()
    return canonical

problem_8_report = apply_transform('T1-CASE-SHIP-CITY', canonical_city_case)
show_repaired_examples('T1-CASE-SHIP-CITY')

{'rule': 'T1-CASE-SHIP-CITY', 'before': 700, 'updated': 700, 'after': 0}
After cleaning examples:
  {'primary_key': {'OrderID': 10249}, 'raw_value': 'münster', 'cleaned_value': 'Münster', 'expected_clean': 'Münster', 'matches_ground_truth': True}
  {'primary_key': {'OrderID': 10294}, 'raw_value': 'ALBUQUERQUE', 'cleaned_value': 'Albuquerque', 'expected_clean': 'Albuquerque', 'matches_ground_truth': True}
  {'primary_key': {'OrderID': 10301}, 'raw_value': 'stuttgart', 'cleaned_value': 'Stuttgart', 'expected_clean': 'Stuttgart', 'matches_ground_truth': True}
  {'primary_key': {'OrderID': 10305}, 'raw_value': 'ANCHORAGE', 'cleaned_value': 'Anchorage', 'expected_clean': 'Anchorage', 'matches_ground_truth': True}
  {'primary_key': {'OrderID': 10360}, 'raw_value': 'strasbourg', 'cleaned_value': 'Strasbourg', 'expected_clean': 'Strasbourg', 'matches_ground_truth': True}


## Problem 9: Country aliases không thống nhất

**Problem.** Customers.Country trộn abbreviation, tên bản địa, lỗi case và khoảng trắng. Một country có thể xuất hiện dưới nhiều label khác nhau.

**Detection.** Chạy code cell tiếp theo để xem tổng số candidate và tối đa 5 ví dụ raw. Candidate detection phục vụ khám phá; exact ground-truth faults mới quyết định các dòng được reference answer sửa.

In [79]:
# Detect Problem 9
problem_9_candidates = show_candidates('T1-COUNTRY-ALIASES', limit=5)

Candidate rows: 59
Exact ground-truth faults: 50
Raw examples:
  {'CustomerID': 'ANATR', 'Country': 'México'}
  {'CustomerID': 'ANTON', 'Country': 'México'}
  {'CustomerID': 'AROUT', 'Country': 'United Kingdom'}
  {'CustomerID': 'BERGS', 'Country': 'sweden'}
  {'CustomerID': 'BLAUS', 'Country': 'DE'}


### Solution 9

Xây vocabulary mapping rõ ràng và map từng alias sang canonical label của dataset.

In [80]:
country_aliases = {
    'México': 'Mexico', 'United Kingdom': 'UK', 'sweden': 'Sweden',
    'DE': 'Germany', 'FRANCE ': 'France', 'España': 'Spain',
    'CA': 'Canada', 'switzerland': 'Switzerland', 'austria': 'Austria',
    'Brasil': 'Brazil', 'U.S.A.': 'USA', 'Venez.': 'Venezuela',
    'belgium': 'Belgium', 'portugal': 'Portugal', 'argentina': 'Argentina',
    'italy': 'Italy', 'norway': 'Norway', 'denmark': 'Denmark',
    'finland': 'Finland', 'poland': 'Poland'
}

def canonical_country(value, record):
    return country_aliases[str(value)]

problem_9_report = apply_transform('T1-COUNTRY-ALIASES', canonical_country)
show_repaired_examples('T1-COUNTRY-ALIASES')

{'rule': 'T1-COUNTRY-ALIASES', 'before': 50, 'updated': 50, 'after': 0}
After cleaning examples:
  {'primary_key': {'CustomerID': 'ANATR'}, 'raw_value': 'México', 'cleaned_value': 'Mexico', 'expected_clean': 'Mexico', 'matches_ground_truth': True}
  {'primary_key': {'CustomerID': 'ANTON'}, 'raw_value': 'México', 'cleaned_value': 'Mexico', 'expected_clean': 'Mexico', 'matches_ground_truth': True}
  {'primary_key': {'CustomerID': 'AROUT'}, 'raw_value': 'United Kingdom', 'cleaned_value': 'UK', 'expected_clean': 'UK', 'matches_ground_truth': True}
  {'primary_key': {'CustomerID': 'BERGS'}, 'raw_value': 'sweden', 'cleaned_value': 'Sweden', 'expected_clean': 'Sweden', 'matches_ground_truth': True}
  {'primary_key': {'CustomerID': 'BLAUS'}, 'raw_value': 'DE', 'cleaned_value': 'Germany', 'expected_clean': 'Germany', 'matches_ground_truth': True}


## Problem 10: ShipPostalCode bị thay bằng NULL

**Problem.** Orders.ShipPostalCode có cả NULL hợp lệ từ baseline và 900 NULL được chèn. Không thể fill toàn bộ NULL bằng một giá trị chung và cũng không thể biết chắc postal code đã bị xóa chỉ từ ô NULL.

**Detection.** Chạy code cell tiếp theo để xem tổng số candidate và tối đa 5 ví dụ raw. Candidate detection phục vụ khám phá; exact ground-truth faults mới quyết định các dòng được reference answer sửa.

In [81]:
# Detect Problem 10
problem_10_candidates = show_candidates('T1-MISSING-SHIP-POSTAL', limit=5)

Candidate rows: 1,072
Exact ground-truth faults: 900
Raw examples:
  {'OrderID': 10249, 'CustomerID': 'TOMSP', 'ShipCity': 'Münster', 'ShipCountry': 'Germany', 'ShipPostalCode': None}
  {'OrderID': 10268, 'CustomerID': 'GROSR', 'ShipCity': 'Caracas', 'ShipCountry': 'Venezuela', 'ShipPostalCode': None}
  {'OrderID': 10270, 'CustomerID': 'WARTH', 'ShipCity': 'Oulu', 'ShipCountry': 'Finland', 'ShipPostalCode': None}
  {'OrderID': 10279, 'CustomerID': 'LEHMS', 'ShipCity': 'Frankfurt a.M.', 'ShipCountry': 'Germany', 'ShipPostalCode': None}
  {'OrderID': 10281, 'CustomerID': 'ROMEY', 'ShipCity': 'Madrid', 'ShipCountry': 'Spain', 'ShipPostalCode': None}


### Solution 10

Dùng record identity trong JSON để phân biệt injected NULL khỏi legitimate NULL, rồi phục hồi exact supervised label. Clean controls bảo đảm NULL hợp lệ không bị sửa.

In [82]:
problem_10_report = restore_supervised_labels(
    'T1-MISSING-SHIP-POSTAL',
    'NULL không chứa postal code đã bị xóa; exact supervised labels là bắt buộc.'
)
show_repaired_examples('T1-MISSING-SHIP-POSTAL')

{'rule': 'T1-MISSING-SHIP-POSTAL', 'method': 'supervised_ground_truth', 'reason': 'NULL không chứa postal code đã bị xóa; exact supervised labels là bắt buộc.', 'before': 900, 'updated': 900, 'after': 0}
After cleaning examples:
  {'primary_key': {'OrderID': 10249}, 'raw_value': None, 'cleaned_value': '44087', 'expected_clean': '44087', 'matches_ground_truth': True}
  {'primary_key': {'OrderID': 10268}, 'raw_value': None, 'cleaned_value': '1081', 'expected_clean': '1081', 'matches_ground_truth': True}
  {'primary_key': {'OrderID': 10270}, 'raw_value': None, 'cleaned_value': '90110', 'expected_clean': '90110', 'matches_ground_truth': True}
  {'primary_key': {'OrderID': 10279}, 'raw_value': None, 'cleaned_value': '60528', 'expected_clean': '60528', 'matches_ground_truth': True}
  {'primary_key': {'OrderID': 10281}, 'raw_value': None, 'cleaned_value': '28001', 'expected_clean': '28001', 'matches_ground_truth': True}


## Problem 11: Quantity bị nhân 100

**Problem.** 800 Order Details.Quantity đã bị nhân 100. Threshold Quantity từ 100 trở lên tạo candidate set có recall đầy đủ nhưng vẫn bao gồm một số giá trị sạch hợp lệ. Vì vậy không được sửa tất cả candidate một cách mù quáng.

**Detection.** Chạy code cell tiếp theo để xem tổng số candidate và tối đa 5 ví dụ raw. Candidate detection phục vụ khám phá; exact ground-truth faults mới quyết định các dòng được reference answer sửa.

In [83]:
# Detect Problem 11
problem_11_candidates = show_candidates('T1-QUANTITY-OUTLIER', limit=5)

Candidate rows: 823
Exact ground-truth faults: 800
Raw examples:
  {'OrderID': 10286, 'ProductID': 35, 'Quantity': 100}
  {'OrderID': 10364, 'ProductID': 69, 'Quantity': 3000}
  {'OrderID': 10398, 'ProductID': 55, 'Quantity': 120}
  {'OrderID': 10451, 'ProductID': 55, 'Quantity': 120}
  {'OrderID': 10452, 'ProductID': 44, 'Quantity': 100}


### Solution 11

Chỉ áp dụng phép chia 100 cho exact affected rows trong training labels và kiểm tra số raw chia hết cho 100. Clean controls đo việc bảo toàn quantity hợp lệ.

In [84]:
def undo_quantity_multiplier(value, record):
    number = int(value)
    assert number % 100 == 0
    return number // 100

problem_11_report = apply_transform('T1-QUANTITY-OUTLIER', undo_quantity_multiplier)
show_repaired_examples('T1-QUANTITY-OUTLIER')

{'rule': 'T1-QUANTITY-OUTLIER', 'before': 800, 'updated': 800, 'after': 0}
After cleaning examples:
  {'primary_key': {'OrderID': 10364, 'ProductID': 69}, 'raw_value': 3000, 'cleaned_value': 30, 'expected_clean': 30, 'matches_ground_truth': True}
  {'primary_key': {'OrderID': 10792, 'ProductID': 68}, 'raw_value': 1500, 'cleaned_value': 15, 'expected_clean': 15, 'matches_ground_truth': True}
  {'primary_key': {'OrderID': 11083, 'ProductID': 12}, 'raw_value': 1100, 'cleaned_value': 11, 'expected_clean': 11, 'matches_ground_truth': True}
  {'primary_key': {'OrderID': 11121, 'ProductID': 15}, 'raw_value': 4000, 'cleaned_value': 40, 'expected_clean': 40, 'matches_ground_truth': True}
  {'primary_key': {'OrderID': 11129, 'ProductID': 2}, 'raw_value': 4700, 'cleaned_value': 47, 'expected_clean': 47, 'matches_ground_truth': True}


## Problem 12: Customer bị clone dưới technical ID mới

**Problem.** 20 customer rows được clone với ID dạng D0001…D0020 nên primary key vẫn hợp lệ. Duplicate phải được phát hiện bằng ID anomaly kết hợp business identity, không chỉ dựa vào primary key.

**Detection.** Chạy code cell tiếp theo để xem tổng số candidate và tối đa 5 ví dụ raw. Candidate detection phục vụ khám phá; exact ground-truth faults mới quyết định các dòng được reference answer sửa.

In [85]:
# Detect Problem 12
problem_12_candidates = show_candidates('T1-DUPLICATE-CUSTOMER', limit=5)

Candidate rows: 20
Exact ground-truth faults: 20
Raw examples:
  {'CustomerID': 'D0001', 'CompanyName': 'Berglunds snabbköp', 'ContactName': 'Christina Berglund'}
  {'CustomerID': 'D0002', 'CompanyName': 'Blauer See Delikatessen', 'ContactName': 'Hanna Moos'}
  {'CustomerID': 'D0003', 'CompanyName': 'Drachenblut Delikatessen', 'ContactName': 'Sven Ottlieb'}
  {'CustomerID': 'D0004', 'CompanyName': 'Franchi S.p.A.', 'ContactName': 'Paolo Accorti'}
  {'CustomerID': 'D0005', 'CompanyName': 'Hanari Carnes', 'ContactName': 'Mario Pontes'}


### Solution 12

Review candidate rows và chỉ DELETE những row được manifest ghi là insert_row. Không dùng broad DELETE theo CompanyName vì có thể xóa nhầm bản ghi gốc.

In [86]:
rule_id = 'T1-DUPLICATE-CUSTOMER'
before = unresolved_count(rule_id)
deleted = 0
for record in records_for(rule_id):
    where, parameters = primary_key_filter(record['primary_key'])
    deleted += connection.execute(
        f'DELETE FROM {quote_name(record["table"])} WHERE {where}', parameters
    ).rowcount
problem_12_report = {
    'rule': rule_id, 'before': before, 'deleted': deleted,
    'after': unresolved_count(rule_id)
}
print(problem_12_report)
show_repaired_examples('T1-DUPLICATE-CUSTOMER')

{'rule': 'T1-DUPLICATE-CUSTOMER', 'before': 20, 'deleted': 20, 'after': 0}
After cleaning examples:
  {'primary_key': {'CustomerID': 'D0001'}, 'raw_value': {'CustomerID': 'D0001', 'CompanyName': 'Berglunds snabbköp', 'ContactName': 'Christina Berglund', 'ContactTitle': 'Order Administrator', 'Address': 'Berguvsvägen  8', 'City': 'Luleå', 'Region': 'Northern Europe', 'PostalCode': 'S-958 22', 'Country': 'sweden', 'Phone': '0921-12 34 65', 'Fax': '0921-12 34 67'}, 'cleaned_value': '<row deleted>', 'expected_clean': None, 'matches_ground_truth': True}
  {'primary_key': {'CustomerID': 'D0002'}, 'raw_value': {'CustomerID': 'D0002', 'CompanyName': 'Blauer See Delikatessen', 'ContactName': 'Hanna Moos', 'ContactTitle': 'Sales Representative', 'Address': 'Forsterstr. 57', 'City': 'Mannheim', 'Region': 'Western Europe', 'PostalCode': '68306', 'Country': 'DE', 'Phone': '0621-08460', 'Fax': '0621-08924'}, 'cleaned_value': '<row deleted>', 'expected_clean': None, 'matches_ground_truth': True}
  {'

## Final validation

Sau 12 solutions, ground-truth faults phải bằng 0 và clean controls phải giữ nguyên. Fingerprint so sánh giá trị của từng bảng theo primary-key order, vì SQLite files có thể khác binary layout dù dữ liệu logic giống nhau.

Validation hoàn toàn tự chứa trong folder: expected fingerprints đã nằm trong JSON, không cần import code hoặc mở database bên ngoài.

In [87]:
def primary_key_columns(table):
    rows = connection.execute(f'PRAGMA table_info({quote_name(table)})').fetchall()
    return [row['name'] for row in sorted(
        (row for row in rows if row['pk']), key=lambda row: row['pk'])]

def table_fingerprint(table):
    keys = primary_key_columns(table)
    query = f'SELECT * FROM {quote_name(table)}'
    if keys:
        query += ' ORDER BY ' + ', '.join(quote_name(key) for key in keys)
    digest = hashlib.sha256()
    for row in connection.execute(query):
        values = [
            {'blob_sha256': hashlib.sha256(value).hexdigest(), 'bytes': len(value)}
            if isinstance(value, bytes) else value
            for value in row
        ]
        line = json.dumps(values, ensure_ascii=False, sort_keys=True,
                          separators=(',', ':'))
        digest.update(line.encode('utf-8') + b'\n')
    return digest.hexdigest()

def score_ground_truth():
    repaired = remaining = preserved = false_positive = 0
    for record in bundle['repair_records']:
        row = find_row(record['table'], record['primary_key'])
        if record['operation'] == 'insert_row':
            repaired += int(row is None)
            remaining += int(row is not None)
        elif row is not None and row[record['column']] == record['clean_value']:
            repaired += 1
        else:
            remaining += 1
    for control in bundle['clean_controls']:
        row = find_row(control['table'], control['primary_key'])
        if row is not None and row[control['column']] == control['clean_value']:
            preserved += 1
        else:
            false_positive += 1
    total = repaired + remaining + preserved + false_positive
    return {
        'quality_score': round(100 * (repaired + preserved) / total, 4),
        'repaired_faults': repaired,
        'remaining_faults': remaining,
        'preserved_clean_controls': preserved,
        'false_positive_controls': false_positive,
    }

# Restore the expected AUTOINCREMENT bookkeeping after deleting injected rows.
connection.execute('DELETE FROM sqlite_sequence')
connection.executemany(
    'INSERT INTO sqlite_sequence(name, seq) VALUES (?, ?)',
    [(item['name'], item['seq'])
     for item in bundle['validation_expectations']['sqlite_sequence']]
)
connection.commit()

score = score_ground_truth()
expected_tables = bundle['validation_expectations']['tables']
matching_tables = 0
for table, expected in expected_tables.items():
    rows = connection.execute(f'SELECT COUNT(*) FROM {quote_name(table)}').fetchone()[0]
    matched = rows == expected['rows'] and table_fingerprint(table) == expected['fingerprint']
    matching_tables += int(matched)

final_validation = {
    **score,
    'integrity_check': connection.execute('PRAGMA integrity_check').fetchone()[0],
    'foreign_key_violations': len(connection.execute('PRAGMA foreign_key_check').fetchall()),
    'matching_tables': matching_tables,
    'table_count': len(expected_tables),
}
final_validation['passed'] = (
    final_validation['quality_score'] == 100.0
    and final_validation['remaining_faults'] == 0
    and final_validation['false_positive_controls'] == 0
    and final_validation['integrity_check'] == 'ok'
    and final_validation['foreign_key_violations'] == 0
    and matching_tables == len(expected_tables)
)
print(json.dumps(final_validation, indent=2))
assert final_validation['passed']

{
  "quality_score": 100.0,
  "repaired_faults": 5170,
  "remaining_faults": 0,
  "preserved_clean_controls": 5170,
  "false_positive_controls": 0,
  "integrity_check": "ok",
  "foreign_key_violations": 0,
  "matching_tables": 13,
  "table_count": 13,
  "passed": true
}


## Publish `northwind_cleaned1.db`

Chỉ khi validation đạt toàn bộ điều kiện, cell cuối mới backup working database ra temporary file rồi thay thế clean output. Raw database vẫn luôn được giữ nguyên.

In [88]:
descriptor, temporary_name = tempfile.mkstemp(
    prefix='.notebook_clean_', suffix='.db', dir=FOLDER
)
os.close(descriptor)
temporary_path = Path(temporary_name)
try:
    destination = sqlite3.connect(temporary_path)
    connection.backup(destination)
    destination.close()
    os.replace(temporary_path, CLEAN)
finally:
    temporary_path.unlink(missing_ok=True)

print('Published:', CLEAN.name)
print('Output SHA-256:', sha256_file(CLEAN))
print('Raw remained unchanged:', sha256_file(RAW) == bundle['dataset']['raw_sha256'])
connection.close()

Published: northwind_cleaned1.db
Output SHA-256: 05268cfa1011bce8c832a8efabf6dfa82eaf16f759f5d437d84fc1ffe05900e5
Raw remained unchanged: True


## Hoàn thành

Bạn đã đi từ raw profiling qua 12 problem/solution cases và sinh clean database. Khi dùng làm training package, AI Agent nên liên kết ba biểu diễn của cùng kiến thức:

- JSON cung cấp specification và exact labels.
- Notebook giải thích reasoning và cho thấy code từng bước.
- Python là reference answer chạy end-to-end bằng một lệnh.